# Notebook 20 — Conditional VAE Inverse Design (A2 CNN + B2)

**Goal**: Train a **conditional Variational Autoencoder (cVAE)** that, given a 1000-point target absorption
spectrum, generates 20 design parameters.

| Aspect | Detail |
|--------|--------|
| **Forward surrogate** | A2 CNN (frozen, from Notebook 18) |
| **Inverse method** | cVAE with spectrum conditioning |
| **Encoder** | (spectrum + params) → μ, log σ² (latent dim 64) |
| **Decoder** | (z + spectrum) → 20 params |
| **Loss** | Reconstruction + β·KL + λ·Spectral MSE (through surrogate) |
| **Key advantage** | Fast single-pass inference, handles multi-modality via latent sampling |

In [ ]:
# ============================================================
# Cell 1 — Imports & Configuration
# ============================================================
import sys, os, time, pickle, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

sys.path.insert(0, os.path.abspath('../src'))
from Theoretical_model import calculate_acoustic_properties
from physics_guided_CD_FiLM import PARAM_RANGES, validate_and_clip_parameters

assert torch.cuda.is_available(), 'CUDA required'
device = torch.device('cuda')
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')
print(f'Device: {torch.cuda.get_device_name()}')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Paths
DATA_PATH      = '../data/lhs_data_full_spectrum.npz'
MODEL_DIR      = '../models'
SURROGATE_PATH = os.path.join(MODEL_DIR, 'forward_surrogate_cnn.pth')
SCALER_PATH    = os.path.join(MODEL_DIR, 'forward_surrogate_cnn_scaler.pkl')
CVAE_PATH      = os.path.join(MODEL_DIR, 'inverse_cvae_a2.pth')
CVAE_SCALER    = os.path.join(MODEL_DIR, 'inverse_cvae_a2_scaler.pkl')

# Hyper-parameters
BATCH_SIZE  = 512
EPOCHS      = 50
LR_INIT     = 1e-3
LR_MIN      = 1e-5
HIDDEN_DIM  = 512
LATENT_DIM  = 64
NUM_PARAMS  = 20
NUM_FREQ    = 1000
BETA_MAX    = 1.0        # KL weight (annealed)
BETA_ANNEAL = 10         # epochs to anneal β from 0→1
LAMBDA_SPEC = 1.0        # spectral MSE weight
NUM_CANDIDATES = 20

PARAM_NAMES = ['d1','d2','d3','d4','d5','d6','d7','d8','d9','d10',
               'm2','m3','m5','m6','m8','m9','rho','eta','E','nu']

print('\n✓ Imports & config ready')

In [ ]:
# ============================================================
# Cell 2 — Load Data & Split
# ============================================================
print('Loading NPZ …')
t0 = time.time()
raw = np.load(DATA_PATH)
params_all  = raw['params'].astype(np.float32)
spectra_all = raw['spectra'].astype(np.float32)
frequencies = raw['frequencies']
print(f'  Loaded in {time.time()-t0:.1f}s')

X_train, X_temp, Y_train, Y_temp = train_test_split(
    params_all, spectra_all, test_size=0.2, random_state=SEED)
X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp, Y_temp, test_size=0.5, random_state=SEED)

print(f'  Train: {X_train.shape[0]:,}  Val: {X_val.shape[0]:,}  Test: {X_test.shape[0]:,}')

scaler_x = MinMaxScaler()
X_train_n = scaler_x.fit_transform(X_train).astype(np.float32)
X_val_n   = scaler_x.transform(X_val).astype(np.float32)
X_test_n  = scaler_x.transform(X_test).astype(np.float32)

# DataLoaders: (spectrum, normalised_params)
def make_loader(spec, params_n, bs, shuffle=True):
    ds = TensorDataset(torch.tensor(spec, dtype=torch.float32),
                       torch.tensor(params_n, dtype=torch.float32))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=0, pin_memory=True)

train_loader = make_loader(Y_train, X_train_n, BATCH_SIZE)
val_loader   = make_loader(Y_val,   X_val_n,   BATCH_SIZE, shuffle=False)
test_loader  = make_loader(Y_test,  X_test_n,  BATCH_SIZE, shuffle=False)
print('✓ Data ready')

In [ ]:
# ============================================================
# Cell 3 — Load A2 CNN Forward Surrogate (frozen)
# ============================================================
class ForwardSurrogateCNN(nn.Module):
    def __init__(self, in_dim=20, out_dim=1000):
        super().__init__()
        self.in_dim = in_dim; self.out_dim = out_dim
        self.stem = nn.Sequential(
            nn.Linear(in_dim, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.01, inplace=True),
            nn.Linear(256, 128*32), nn.BatchNorm1d(128*32), nn.LeakyReLU(0.01, inplace=True))
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(128, 64, 4, 2, 1), nn.BatchNorm1d(64), nn.LeakyReLU(0.01, inplace=True),
            nn.ConvTranspose1d(64, 32, 4, 2, 1), nn.BatchNorm1d(32), nn.LeakyReLU(0.01, inplace=True),
            nn.ConvTranspose1d(32, 16, 4, 2, 1), nn.BatchNorm1d(16), nn.LeakyReLU(0.01, inplace=True),
            nn.ConvTranspose1d(16, 8, 4, 2, 1), nn.BatchNorm1d(8), nn.LeakyReLU(0.01, inplace=True))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(8*512, 1024),
            nn.LeakyReLU(0.01, inplace=True), nn.Linear(1024, out_dim), nn.Sigmoid())
    def forward(self, x):
        h = self.stem(x); h = h.view(-1, 128, 32); h = self.decoder(h); return self.head(h)

ckpt_surr = torch.load(SURROGATE_PATH, map_location=device, weights_only=False)
arch = ckpt_surr['architecture']
surrogate = ForwardSurrogateCNN(**arch).to(device)
surrogate.load_state_dict(ckpt_surr['model_state_dict'])
surrogate.eval()
for p in surrogate.parameters(): p.requires_grad = False
print(f'✓ A2 CNN Surrogate loaded (MSE={ckpt_surr["test_metrics"]["spectral_mse"]:.2e})')

## cVAE Architecture

**Encoder** (used during training):
```
[SpectrumEncoder(1000→512) ‖ ParamMLP(20→256)] → concat(768)
  → MLP(768→256) → (μ, log_σ²) each dim 64
```

**Decoder** (used at inference):
```
[z(64) ‖ SpectrumEncoder(1000→512)] → concat(576)
  → MLP(576→256→256→20) → Sigmoid
```

In [ ]:
# ============================================================
# Cell 4 — cVAE Model Definition
# ============================================================
class SpectrumEncoder(nn.Module):
    """1D-CNN: 1000-point spectrum → embedding."""
    def __init__(self, embed_dim=512):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32,  kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(32), nn.LeakyReLU(0.01, inplace=True),
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(64), nn.LeakyReLU(0.01, inplace=True),
            nn.Conv1d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(128), nn.LeakyReLU(0.01, inplace=True),
            nn.AdaptiveAvgPool1d(1),
        )
        self.fc = nn.Sequential(nn.Linear(128, embed_dim), nn.LeakyReLU(0.01, inplace=True))

    def forward(self, spectrum):
        x = spectrum.unsqueeze(1)
        x = self.conv(x).squeeze(-1)
        return self.fc(x)


class ConditionalVAE(nn.Module):
    def __init__(self, num_params=20, num_freq=1000, latent_dim=64,
                 spec_embed_dim=512, param_embed_dim=256):
        super().__init__()
        self.latent_dim = latent_dim

        # Shared spectrum encoder
        self.spec_enc = SpectrumEncoder(embed_dim=spec_embed_dim)

        # --- ENCODER ---
        self.param_enc = nn.Sequential(
            nn.Linear(num_params, param_embed_dim),
            nn.LeakyReLU(0.01, inplace=True),
        )
        enc_in = spec_embed_dim + param_embed_dim  # 768
        self.enc_fc = nn.Sequential(
            nn.Linear(enc_in, 256), nn.LeakyReLU(0.01, inplace=True),
        )
        self.fc_mu     = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)

        # --- DECODER ---
        dec_in = latent_dim + spec_embed_dim  # 576
        self.decoder = nn.Sequential(
            nn.Linear(dec_in, 256),
            nn.BatchNorm1d(256), nn.LeakyReLU(0.01, inplace=True),
            nn.Linear(256, 256),
            nn.BatchNorm1d(256), nn.LeakyReLU(0.01, inplace=True),
            nn.Linear(256, num_params),
            nn.Sigmoid(),
        )

    def encode(self, params_norm, spectrum):
        s_emb = self.spec_enc(spectrum)
        p_emb = self.param_enc(params_norm)
        h = self.enc_fc(torch.cat([s_emb, p_emb], dim=-1))
        return self.fc_mu(h), self.fc_logvar(h), s_emb

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, spec_emb):
        return self.decoder(torch.cat([z, spec_emb], dim=-1))

    def forward(self, params_norm, spectrum):
        mu, logvar, s_emb = self.encode(params_norm, spectrum)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z, s_emb)
        return recon, mu, logvar

    @torch.no_grad()
    def sample(self, spectrum, num_samples=1):
        """Generate params from prior z ~ N(0,I) conditioned on spectrum."""
        s_emb = self.spec_enc(spectrum)  # (1, 512) or (B, 512)
        if num_samples > 1 and s_emb.size(0) == 1:
            s_emb = s_emb.expand(num_samples, -1)
        z = torch.randn(s_emb.size(0), self.latent_dim, device=s_emb.device)
        return self.decode(z, s_emb)


cvae = ConditionalVAE(
    num_params=NUM_PARAMS, num_freq=NUM_FREQ, latent_dim=LATENT_DIM,
    spec_embed_dim=HIDDEN_DIM, param_embed_dim=256
).to(device)

total_p = sum(p.numel() for p in cvae.parameters())
print(f'✓ ConditionalVAE — {total_p:,} parameters (latent_dim={LATENT_DIM})')

In [ ]:
# ============================================================
# Cell 5 — Loss, Optimiser, Scheduler
# ============================================================
def build_freq_weights(num_freq=1000, low_cutoff=400, low_weight=2.0, device='cpu'):
    w = torch.ones(num_freq, device=device)
    w[:low_cutoff] = low_weight
    return w / w.mean()

freq_w = build_freq_weights(device=device)

def cvae_loss(recon, target_params, mu, logvar, target_spec, surrogate_fn,
              beta=1.0, lam_spec=1.0):
    """
    L = L_recon + β · D_KL + λ · L_spectral
    """
    # Reconstruction loss (parameter space)
    L_recon = F.mse_loss(recon, target_params)

    # KL divergence
    L_kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    # Spectral loss through frozen surrogate
    pred_spec = surrogate_fn(recon)
    L_spec = ((pred_spec - target_spec)**2 * freq_w).mean()

    total = L_recon + beta * L_kl + lam_spec * L_spec
    return total, L_recon.item(), L_kl.item(), L_spec.item()


optimizer = optim.Adam(cvae.parameters(), lr=LR_INIT, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
print('✓ Loss & optimiser configured')

In [ ]:
# ============================================================
# Cell 6 — Training Loop
# ============================================================
train_losses, val_losses = [], []
train_kl, val_kl = [], []
best_val = float('inf')
best_state = None

print(f'Training {EPOCHS} epochs …\n')
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    beta = min(BETA_MAX, BETA_MAX * epoch / BETA_ANNEAL)   # β annealing

    # ----- Train -----
    cvae.train()
    ep_loss, ep_kl = 0.0, 0.0
    for spec_b, params_b in train_loader:
        spec_b, params_b = spec_b.to(device), params_b.to(device)
        recon, mu, logvar = cvae(params_b, spec_b)
        loss, l_rec, l_kl, l_spec = cvae_loss(
            recon, params_b, mu, logvar, spec_b, surrogate, beta, LAMBDA_SPEC)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(cvae.parameters(), max_norm=1.0)
        optimizer.step()

        bs = spec_b.size(0)
        ep_loss += loss.item() * bs
        ep_kl   += l_kl * bs

    train_losses.append(ep_loss / len(train_loader.dataset))
    train_kl.append(ep_kl / len(train_loader.dataset))

    # ----- Validate -----
    cvae.eval()
    v_loss, v_kl = 0.0, 0.0
    with torch.no_grad():
        for spec_b, params_b in val_loader:
            spec_b, params_b = spec_b.to(device), params_b.to(device)
            recon, mu, logvar = cvae(params_b, spec_b)
            loss, _, l_kl, _ = cvae_loss(
                recon, params_b, mu, logvar, spec_b, surrogate, beta, LAMBDA_SPEC)
            v_loss += loss.item() * spec_b.size(0)
            v_kl   += l_kl * spec_b.size(0)

    val_losses.append(v_loss / len(val_loader.dataset))
    val_kl.append(v_kl / len(val_loader.dataset))
    scheduler.step()

    if val_losses[-1] < best_val:
        best_val   = val_losses[-1]
        best_state = {k: v.cpu().clone() for k, v in cvae.state_dict().items()}
        tag = ' ★'
    else:
        tag = ''

    if epoch % 5 == 0 or epoch == 1:
        lr = scheduler.get_last_lr()[0]
        print(f'  Epoch {epoch:3d}/{EPOCHS}  train {train_losses[-1]:.6f}  '
              f'val {val_losses[-1]:.6f}  KL {val_kl[-1]:.4f}  β {beta:.2f}  lr {lr:.2e}{tag}')

elapsed = time.time() - t_start
print(f'\n✓ Training complete in {elapsed/60:.1f} min — best val = {best_val:.6f}')
cvae.load_state_dict(best_state)
cvae.to(device)
print('  Best checkpoint restored')

In [ ]:
# ============================================================
# Cell 7 — Learning Curves
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.semilogy(range(1, EPOCHS+1), train_losses, label='Train')
ax1.semilogy(range(1, EPOCHS+1), val_losses,   label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Total Loss (log)')
ax1.set_title('cVAE Total Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(range(1, EPOCHS+1), train_kl, label='Train KL')
ax2.plot(range(1, EPOCHS+1), val_kl,   label='Val KL')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('KL Divergence')
ax2.set_title('KL Divergence'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('cVAE (A2+B2) — Learning Curves', fontsize=13)
plt.tight_layout(); plt.show()

## Test-Set Evaluation

In [ ]:
# ============================================================
# Cell 8 — Test Metrics via Surrogate
# ============================================================
cvae.eval()
all_spec_mse = []

with torch.no_grad():
    for spec_b, params_b in test_loader:
        spec_b = spec_b.to(device)
        # Sample from prior (no encoder at inference)
        pred_params = cvae.sample(spec_b, num_samples=1)  # uses 1 sample per input
        recon_spec  = surrogate(pred_params)
        mse = ((recon_spec - spec_b)**2).mean(dim=1)  # per sample
        all_spec_mse.append(mse.cpu().numpy())

spec_mse_arr = np.concatenate(all_spec_mse)
print('═══════════════════════════════════════════')
print('  TEST METRICS — cVAE (A2+B2), single sample')
print('═══════════════════════════════════════════')
print(f'  Mean spectral MSE   : {spec_mse_arr.mean():.6e}')
print(f'  Median spectral MSE : {np.median(spec_mse_arr):.6e}')
for p in [90, 95, 99]:
    print(f'  {p}th pctile MSE    : {np.percentile(spec_mse_arr, p):.6e}')
print('═══════════════════════════════════════════')

In [ ]:
# ============================================================
# Cell 9 — Best-of-N Test (20 candidates per target)
# ============================================================
np.random.seed(123)
bon_idx = np.random.choice(len(X_test), 200, replace=False)

bon_mse = []
cvae.eval()
with torch.no_grad():
    for idx in bon_idx:
        spec_t = torch.tensor(Y_test[idx], dtype=torch.float32, device=device).unsqueeze(0)
        # Generate N candidates
        cands = cvae.sample(spec_t, num_samples=NUM_CANDIDATES)  # (N, 20)
        specs = surrogate(cands)  # (N, 1000)
        mses  = ((specs - spec_t)**2).mean(dim=1)
        bon_mse.append(mses.min().item())

bon_mse = np.array(bon_mse)
print(f'Best-of-{NUM_CANDIDATES} test (200 targets):')
print(f'  Mean MSE : {bon_mse.mean():.6e}')
print(f'  Median   : {np.median(bon_mse):.6e}')
print(f'  95th pct : {np.percentile(bon_mse, 95):.6e}')

In [ ]:
# ============================================================
# Cell 10 — Overlay: 6 Spectra
# ============================================================
np.random.seed(77)
show_idx = np.random.choice(len(X_test), 6, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True, sharey=True)
cvae.eval()
for ax, idx in zip(axes.flat, show_idx):
    target = Y_test[idx]
    spec_t = torch.tensor(target, dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        cands = cvae.sample(spec_t, num_samples=NUM_CANDIDATES)
        specs = surrogate(cands)
        mses  = ((specs - spec_t)**2).mean(dim=1)
        best  = specs[mses.argmin()].cpu().numpy()

    ax.plot(frequencies, target, 'b-',  lw=1.2, label='Target')
    ax.plot(frequencies, best,   'r--', lw=1.0, label='Best cVAE')
    ax.set_title(f'MSE={mses.min().item():.2e}', fontsize=10)
    ax.set_ylim(-0.02, 1.02); ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

fig.supxlabel('Frequency (Hz)')
fig.supylabel('Absorption Coefficient')
fig.suptitle('cVAE (A2+B2) — Best-of-20 Reconstruction', fontsize=13)
plt.tight_layout(); plt.show()

## TMM Validation

In [ ]:
# ============================================================
# Cell 11 — TMM Validation (5 samples)
# ============================================================
NUM_TMM = 5
np.random.seed(99)
tmm_idx = np.random.choice(len(X_test), NUM_TMM, replace=False)

fig, axes = plt.subplots(1, NUM_TMM, figsize=(20, 4), sharex=True, sharey=True)
tmm_results = []

for ax, idx in zip(axes, tmm_idx):
    target = Y_test[idx]
    spec_t = torch.tensor(target, dtype=torch.float32, device=device).unsqueeze(0)

    with torch.no_grad():
        cands = cvae.sample(spec_t, num_samples=NUM_CANDIDATES)
        specs = surrogate(cands)
        mses  = ((specs - spec_t)**2).mean(dim=1)
        best_n = cands[mses.argmin()].cpu().numpy()

    best_raw = scaler_x.inverse_transform(best_n.reshape(1, -1)).flatten()
    best_raw = validate_and_clip_parameters(best_raw.reshape(1, -1)).flatten()

    param_dict = {
        'rho': float(best_raw[16]), 'eta': float(best_raw[17]),
        'E': float(best_raw[18]), 'nu': float(best_raw[19]), 'W': 2000.0,
        'd': [float(best_raw[i]) for i in range(10)],
        'm': {2: float(best_raw[10]), 3: float(best_raw[11]),
              5: float(best_raw[12]), 6: float(best_raw[13]),
              8: float(best_raw[14]), 9: float(best_raw[15])}
    }
    _, alpha_tmm, _, _ = calculate_acoustic_properties(param_dict)

    mse_tmm = np.mean((alpha_tmm - target)**2)
    avg_err = abs(alpha_tmm.mean() - target.mean())
    tmm_results.append({'idx': idx, 'mse': mse_tmm, 'avg_err': avg_err})

    ax.plot(frequencies, target,    'b-',  lw=1.2, label='Target')
    ax.plot(frequencies, alpha_tmm, 'r--', lw=1.0, label='TMM(pred θ)')
    ax.set_title(f'MSE={mse_tmm:.2e}\nΔavg={avg_err:.4f}', fontsize=9)
    ax.set_ylim(-0.02, 1.02); ax.grid(True, alpha=0.3)
    if idx == tmm_idx[0]: ax.legend(fontsize=8)

fig.supxlabel('Frequency (Hz)')
fig.suptitle('cVAE (A2+B2) — TMM Validation', fontsize=13)
plt.tight_layout(); plt.show()

print('\n  TMM Validation:')
for r in tmm_results:
    print(f'    Sample {r["idx"]:6d}  MSE={r["mse"]:.4e}  Δavg={r["avg_err"]:.4f}')
print(f'  Mean TMM MSE: {np.mean([r["mse"] for r in tmm_results]):.4e}')

In [ ]:
# ============================================================
# Cell 12 — Save Model
# ============================================================
torch.save({
    'model_state_dict': cvae.state_dict(),
    'architecture': {
        'num_params': NUM_PARAMS, 'num_freq': NUM_FREQ,
        'latent_dim': LATENT_DIM, 'spec_embed_dim': HIDDEN_DIM,
        'param_embed_dim': 256,
    },
    'train_loss': train_losses, 'val_loss': val_losses,
    'best_val_loss': best_val,
    'surrogate_path': SURROGATE_PATH,
}, CVAE_PATH)

with open(CVAE_SCALER, 'wb') as f:
    pickle.dump(scaler_x, f)

print(f'✓ Model saved → {CVAE_PATH}  ({os.path.getsize(CVAE_PATH)/1e6:.1f} MB)')

---
## Summary

| Metric | Value |
|--------|-------|
| Architecture | cVAE: CNN spectrum encoder + latent dim 64 |
| Forward surrogate | A2 CNN (from Notebook 18) |
| Loss | Recon + β·KL (annealed) + λ·Spectral MSE |
| Inference | Single pass (fast) + multi-sample via prior |

**Key advantage:** Fast inference, generates diverse solutions via latent sampling.  
**Limitation:** May suffer mode collapse if KL/β balance is poor.